# Build clip manifest

In [ ]:
from __future__ import annotations

import hashlib
import logging
import re
import sys
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import pandas as pd
import yaml

In [ ]:
# --- Project root setup ---
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
assert CONFIG_PATH.exists(), f"Missing config file: {CONFIG_PATH}"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

def resolve_path(raw_path: str) -> Path:
    path = Path(raw_path)
    return path if path.is_absolute() else (PROJECT_ROOT / path).resolve()

OUTPUT_DIR = resolve_path(CONFIG["paths"]["output_dir"])
LOG_DIR = resolve_path(CONFIG["paths"]["log_dir"])
MANIFEST_DIR = OUTPUT_DIR / "00_manifest"

METADATA_PATH = resolve_path(CONFIG["paths"]["metadata_file"])
DATASET_ZIP = resolve_path(CONFIG["paths"]["dataset_zip"])
EXTRACTED_DIR = resolve_path(CONFIG["paths"]["extracted_audio_dir"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("METADATA_PATH =", METADATA_PATH)
print("DATASET_ZIP   =", DATASET_ZIP)
print("EXTRACTED_DIR =", EXTRACTED_DIR)

In [ ]:
# --- Logging setup ---
LOG_FILE = LOG_DIR / "00_build_manifest.log"

logger = logging.getLogger("stage4_manifest")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

logger.info("Logging initialized")

In [ ]:
@dataclass
class AudioFileRecord:
    audio_path: Path
    audio_relpath: str
    audio_filename: str
    audio_basename: str
    audio_extension: str
    file_size_bytes: int

def clean_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = text.replace("\xa0", " ").replace("\u200b", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_column_name(name: str) -> str:
    text = clean_text(name).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")

def normalize_for_match(value: str) -> str:
    text = clean_text(value).lower().replace("\\", "/")
    text = re.sub(r"/+", "/", text)
    return text.strip("/")

def slugify(value: str, max_length: int = 80) -> str:
    text = clean_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return text[:max_length] if text else "clip"

def stable_short_hash(value: str, length: int = 10) -> str:
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:length]

def safe_extract_zip(zip_path: Path, extract_dir: Path) -> None:
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            member_path = extract_dir / member.filename
            resolved = member_path.resolve()
            if not str(resolved).startswith(str(extract_dir.resolve())):
                raise RuntimeError(f"Unsafe zip entry detected: {member.filename}")
        zf.extractall(extract_dir)

def ensure_audio_source_available(dataset_zip: Path, extracted_dir: Path, auto_extract_zip: bool) -> None:
    extracted_has_content = extracted_dir.exists() and any(extracted_dir.rglob("*"))

    if extracted_has_content:
        logger.info("Using existing extracted audio folder: %s", extracted_dir)
        return

    if not dataset_zip.exists() and not extracted_has_content:
        raise FileNotFoundError(
            f"Neither extracted audio directory nor Dataset.zip is available. Missing zip: {dataset_zip}"
        )

    if not auto_extract_zip and not extracted_has_content:
        raise FileNotFoundError(
            "Extracted audio folder is missing and auto_extract_zip is disabled in config."
        )

    logger.info("Extracting Dataset.zip from %s to %s", dataset_zip, extracted_dir)
    safe_extract_zip(dataset_zip, extracted_dir)
    logger.info("Finished extracting Dataset.zip")

def index_audio_files(audio_root: Path, audio_extensions: Sequence[str]) -> List[AudioFileRecord]:
    allowed = {ext.lower() for ext in audio_extensions}
    records = []
    for path in sorted(audio_root.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix.lower() not in allowed:
            continue
        relpath = path.relative_to(audio_root).as_posix()
        records.append(
            AudioFileRecord(
                audio_path=path.resolve(),
                audio_relpath=relpath,
                audio_filename=path.name,
                audio_basename=path.stem,
                audio_extension=path.suffix.lower(),
                file_size_bytes=path.stat().st_size,
            )
        )
    return records

def build_audio_indexes(records: Sequence[AudioFileRecord]) -> Dict[str, Dict[str, List[AudioFileRecord]]]:
    by_relpath = defaultdict(list)
    by_filename = defaultdict(list)
    by_basename = defaultdict(list)

    for record in records:
        by_relpath[normalize_for_match(record.audio_relpath)].append(record)
        by_filename[normalize_for_match(record.audio_filename)].append(record)
        by_basename[normalize_for_match(record.audio_basename)].append(record)

    return {
        "relative_path": by_relpath,
        "file_name": by_filename,
        "base_name": by_basename,
    }

def score_candidate(metadata_row: pd.Series, candidate: AudioFileRecord, matched_sources: Sequence[str]) -> int:
    score = 0
    relpath_norm = normalize_for_match(metadata_row.get("relative_path", ""))
    filename_norm = normalize_for_match(metadata_row.get("file_name", ""))
    basename_norm = normalize_for_match(metadata_row.get("base_name", ""))
    main_folder_norm = normalize_for_match(metadata_row.get("main_folder", ""))

    if relpath_norm and relpath_norm == normalize_for_match(candidate.audio_relpath):
        score += 100
    if filename_norm and filename_norm == normalize_for_match(candidate.audio_filename):
        score += 40
    if basename_norm and basename_norm == normalize_for_match(candidate.audio_basename):
        score += 30
    if main_folder_norm and main_folder_norm in normalize_for_match(candidate.audio_relpath):
        score += 10
    score += len(set(matched_sources))
    return score

def match_metadata_row(metadata_row: pd.Series, audio_indexes: Dict[str, Dict[str, List[AudioFileRecord]]], priority: Sequence[str]):
    candidate_map = {}
    matched_sources_by_path = defaultdict(list)

    for key in priority:
        raw_value = clean_text(metadata_row.get(key, ""))
        lookup_key = normalize_for_match(raw_value)
        if not lookup_key:
            continue
        hits = audio_indexes[key].get(lookup_key, [])
        for hit in hits:
            path_key = str(hit.audio_path)
            candidate_map[path_key] = hit
            matched_sources_by_path[path_key].append(key)

    if not candidate_map:
        return None, [], "no_audio_candidate", 0

    scored = []
    for path_key, candidate in candidate_map.items():
        sources = matched_sources_by_path[path_key]
        score = score_candidate(metadata_row, candidate, sources)
        scored.append((score, path_key, candidate))

    scored.sort(key=lambda item: (item[0], item[2].audio_relpath), reverse=True)
    best_score, best_path, best_candidate = scored[0]
    best_ties = [item for item in scored if item[0] == best_score]

    if len(best_ties) > 1:
        matched_sources = sorted(set(matched_sources_by_path[best_path]))
        return None, matched_sources, "ambiguous_best_match", best_score

    matched_sources = sorted(set(matched_sources_by_path[best_path]))
    return best_candidate, matched_sources, "matched", best_score

def make_clip_id(metadata_row_number: int, base_name: str) -> str:
    base_slug = slugify(base_name or f"row_{metadata_row_number}")
    base_hash = stable_short_hash(f"{metadata_row_number}:{base_name}")
    return f"clip_{metadata_row_number:04d}_{base_slug}_{base_hash}"

def markdown_table(rows: Sequence[Tuple[str, str]], headers: Tuple[str, str]) -> str:
    output = [f"| {headers[0]} | {headers[1]} |", f"| --- | --- |"]
    for left, right in rows:
        left_clean = clean_text(left).replace("|", "\\|")
        right_clean = clean_text(right).replace("|", "\\|")
        output.append(f"| {left_clean} | {right_clean} |")
    return "\n".join(output)

def build_reality_report(
    metadata_df: pd.DataFrame,
    matched_df: pd.DataFrame,
    unmatched_metadata_df: pd.DataFrame,
    unmatched_audio_df: pd.DataFrame,
    audio_records: Sequence[AudioFileRecord],
    original_to_normalized_cols: Sequence[Tuple[str, str]],
    metadata_path: Path,
    audio_root: Path,
) -> str:
    ext_counts = Counter(record.audio_extension for record in audio_records)
    duplicate_audio_usage = (
        matched_df.groupby("audio_relpath").size().sort_values(ascending=False) if not matched_df.empty else pd.Series(dtype=int)
    )
    duplicate_metadata_files = (
        metadata_df.groupby("file_name").size().sort_values(ascending=False) if "file_name" in metadata_df.columns else pd.Series(dtype=int)
    )
    missing_counts = metadata_df.isna().sum().sort_values(ascending=False)

    report_lines = [
        "# Dataset Reality Report\n",
        "## Inputs\n",
        f"- Metadata file: `{metadata_path}`",
        f"- Audio root indexed: `{audio_root}`",
        f"- Manifest output directory: `{MANIFEST_DIR}`\n",
        "## Metadata summary\n",
        f"- Total metadata rows: **{len(metadata_df)}**",
        f"- Normalized metadata columns: **{len(metadata_df.columns)}**",
        f"- Matched rows written to manifest: **{len(matched_df)}**",
        f"- Unmatched metadata rows: **{len(unmatched_metadata_df)}**",
        f"- Recursively indexed audio files: **{len(audio_records)}**",
        f"- Unmatched audio files: **{len(unmatched_audio_df)}**\n",
        "## Column normalization\n",
        markdown_table(original_to_normalized_cols, headers=("Original column", "Normalized column")),
        "\n\n## Missing values by normalized metadata column\n",
        markdown_table([(str(idx), str(val)) for idx, val in missing_counts.items()], headers=("Column", "Missing count")),
        "\n\n## Audio file inventory by extension\n",
        markdown_table([(ext, str(count)) for ext, count in sorted(ext_counts.items())], headers=("Extension", "Count")),
    ]

    if not duplicate_metadata_files.empty:
        repeated_metadata = duplicate_metadata_files[duplicate_metadata_files > 1].head(10)
        report_lines.extend(
            [
                "\n\n## Repeated metadata file_name values (top 10)\n",
                markdown_table([(str(idx), str(val)) for idx, val in repeated_metadata.items()], headers=("file_name", "Count")),
            ]
        )

    if not duplicate_audio_usage.empty:
        repeated_audio = duplicate_audio_usage[duplicate_audio_usage > 1].head(10)
        report_lines.extend(
            [
                "\n\n## Audio files matched more than once (top 10)\n",
                markdown_table([(str(idx), str(val)) for idx, val in repeated_audio.items()], headers=("audio_relpath", "Count")),
            ]
        )

    if not unmatched_metadata_df.empty:
        cols = [c for c in ["metadata_row_number", "file_name", "base_name", "relative_path", "unmatched_reason"] if c in unmatched_metadata_df.columns]
        preview = unmatched_metadata_df[cols].head(15)
        report_lines.extend(["\n\n## Example unmatched metadata rows\n", preview.to_markdown(index=False)])

    if not unmatched_audio_df.empty:
        cols = [c for c in ["audio_relpath", "audio_filename", "audio_extension", "file_size_bytes"] if c in unmatched_audio_df.columns]
        preview = unmatched_audio_df[cols].head(15)
        report_lines.extend(["\n\n## Example unmatched audio files\n", preview.to_markdown(index=False)])

    report_lines.extend(
        [
            "\n\n## Notes\n",
            "- The metadata workbook is treated as the source of truth for labels and context fields.",
            "- Unmatched rows are written explicitly; no metadata rows are silently discarded.",
            "- This report only reflects fields that actually exist in the workbook loaded for this run.",
        ]
    )
    return "\n".join(report_lines) + "\n"

In [ ]:
# --- Execute manifest build ---
assert METADATA_PATH.exists(), f"Metadata file not found: {METADATA_PATH}"

ensure_audio_source_available(
    dataset_zip=DATASET_ZIP,
    extracted_dir=EXTRACTED_DIR,
    auto_extract_zip=bool(CONFIG["manifest"].get("auto_extract_zip", True)),
)

audio_records = index_audio_files(EXTRACTED_DIR, CONFIG["manifest"]["audio_extensions"])
if not audio_records:
    raise RuntimeError(f"No audio files found under extracted directory: {EXTRACTED_DIR}")

logger.info("Indexed %d audio files under %s", len(audio_records), EXTRACTED_DIR)

raw_metadata_df = pd.read_excel(METADATA_PATH)
original_columns = [str(col) for col in raw_metadata_df.columns]
normalized_columns = [normalize_column_name(col) for col in original_columns]
original_to_normalized_cols = list(zip(original_columns, normalized_columns))
raw_metadata_df.columns = normalized_columns

metadata_df = raw_metadata_df.copy()
for column in metadata_df.columns:
    if metadata_df[column].dtype == object:
        metadata_df[column] = metadata_df[column].map(clean_text)

# Correct known data-entry errors in the source spreadsheet: recorder labels
# "Sennheiser + Zoom F7" ... "F15" are autofill variants of "Sennheiser + Zoom F6",
# and one placement label contains a typo.
if "microphone" in metadata_df.columns:
    metadata_df["microphone"] = metadata_df["microphone"].replace(
        {f"Sennheiser + Zoom F{i}": "Sennheiser + Zoom F6" for i in range(7, 16)})
if "mic_placement_context" in metadata_df.columns:
    metadata_df["mic_placement_context"] = metadata_df["mic_placement_context"].replace(
        {"Besides the resting scetion": "Besides the resting section"})

metadata_df.insert(0, "metadata_row_number", range(2, len(metadata_df) + 2))
metadata_df.insert(1, "metadata_source_file", METADATA_PATH.name)

logger.info("Loaded %d metadata rows from %s", len(metadata_df), METADATA_PATH)
logger.info("Normalized metadata columns: %s", ", ".join(metadata_df.columns))

audio_indexes = build_audio_indexes(audio_records)
priority = CONFIG["manifest"].get("metadata_match_priority", ["relative_path", "file_name", "base_name"])

matched_rows = []
unmatched_metadata_rows = []
matched_audio_relpaths = set()

for _, row in metadata_df.iterrows():
    matched_audio, matched_by, status, score = match_metadata_row(row, audio_indexes, priority)
    row_dict = row.to_dict()
    row_dict["match_status"] = status
    row_dict["match_score"] = score
    row_dict["matched_by"] = ";".join(matched_by)
    row_dict["clip_id"] = make_clip_id(int(row["metadata_row_number"]), clean_text(row.get("base_name", "")))

    if matched_audio is None:
        row_dict["unmatched_reason"] = status
        unmatched_metadata_rows.append(row_dict)
        continue

    row_dict.update(
        {
            "audio_path": str(matched_audio.audio_path),
            "audio_relpath": matched_audio.audio_relpath,
            "audio_filename": matched_audio.audio_filename,
            "audio_basename": matched_audio.audio_basename,
            "audio_extension": matched_audio.audio_extension,
            "file_size_bytes": matched_audio.file_size_bytes,
        }
    )
    matched_rows.append(row_dict)
    matched_audio_relpaths.add(matched_audio.audio_relpath)

matched_df = pd.DataFrame(matched_rows)
unmatched_metadata_df = pd.DataFrame(unmatched_metadata_rows)

unmatched_audio_records = [
    {
        "audio_path": str(record.audio_path),
        "audio_relpath": record.audio_relpath,
        "audio_filename": record.audio_filename,
        "audio_basename": record.audio_basename,
        "audio_extension": record.audio_extension,
        "file_size_bytes": record.file_size_bytes,
    }
    for record in audio_records
    if record.audio_relpath not in matched_audio_relpaths
]
unmatched_audio_df = pd.DataFrame(unmatched_audio_records)

MANIFEST_PATH = MANIFEST_DIR / "manifest.csv"
UNMATCHED_METADATA_PATH = MANIFEST_DIR / "unmatched_metadata_rows.csv"
UNMATCHED_AUDIO_PATH = MANIFEST_DIR / "unmatched_audio_files.csv"
REPORT_PATH = MANIFEST_DIR / "dataset_reality_report.md"

matched_df.to_csv(MANIFEST_PATH, index=False)
unmatched_metadata_df.to_csv(UNMATCHED_METADATA_PATH, index=False)
unmatched_audio_df.to_csv(UNMATCHED_AUDIO_PATH, index=False)

report_text = build_reality_report(
    metadata_df=metadata_df,
    matched_df=matched_df,
    unmatched_metadata_df=unmatched_metadata_df,
    unmatched_audio_df=unmatched_audio_df,
    audio_records=audio_records,
    original_to_normalized_cols=original_to_normalized_cols,
    metadata_path=METADATA_PATH,
    audio_root=EXTRACTED_DIR,
)
REPORT_PATH.write_text(report_text, encoding="utf-8")

logger.info("Manifest written to %s", MANIFEST_PATH)
logger.info("Unmatched metadata rows written to %s", UNMATCHED_METADATA_PATH)
logger.info("Unmatched audio files written to %s", UNMATCHED_AUDIO_PATH)
logger.info("Reality report written to %s", REPORT_PATH)
logger.info(
    "Summary | metadata_rows=%d | matched=%d | unmatched_metadata=%d | audio_indexed=%d | unmatched_audio=%d",
    len(metadata_df),
    len(matched_df),
    len(unmatched_metadata_df),
    len(audio_records),
    len(unmatched_audio_df),
)

print("\n--- Preview: manifest.csv ---")
display(matched_df.head(10))

print("\n--- Preview: unmatched_metadata_rows.csv ---")
display(unmatched_metadata_df.head(10))

print("\n--- Preview: unmatched_audio_files.csv ---")
display(unmatched_audio_df.head(10))